In [2]:
import openai
import instructor

from pydantic import BaseModel, Field
from qdrant_client import QdrantClient

### RAG Pipeline

In [3]:
client = instructor.from_provider(
    "openai/gpt-5.4-nano",
    mode=instructor.Mode.RESPONSES_TOOLS
)

In [4]:
class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")

In [5]:
def get_embedding(text, model='text-embedding-3-small'):

    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):
    
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting.

    Context:
    {preprocessed_context}

    Question:
    {question}
    """

    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=RAGGenerationResponse
)

    return response

def RAG_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocess_context = process_context(retrieved_context)
    prompt = build_prompt(preprocess_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "data_object": answer,
        "answer": answer.answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [6]:
output = RAG_pipeline("Can I buy a fiction book?")

In [7]:
output

{'data_object': RAGGenerationResponse(answer='Yes. The available products include several fiction books, such as The Man His Means & His Methods, Devil of Dublin, Shorn, Monsters of Midlife, and Faultless Love.'),
 'answer': 'Yes. The available products include several fiction books, such as The Man His Means & His Methods, Devil of Dublin, Shorn, Monsters of Midlife, and Faultless Love.',
 'question': 'Can I buy a fiction book?',
 'retrieved_context_ids': ['B0BLFT3P9C',
  'B0BFW7MV1C',
  'B09XGZV7YT',
  'B0B6LSBPR1',
  '1737747227'],
 'retrieved_context': ['The Man His Means & His Methods The hardcover version of this beautiful novel includes over 75 original two-page spread, full-color works of art, designed by the author with AI . (Paperback is black and white with one page chapter art.) Conspiracy theories meet science FACTion in this novel about all the topics we have been discouraged from questioning! Questions like... What if inter-dimensional entities and extra-terrestrial bein

### RAG Pipeline with grounding context

In [8]:
class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question.")
    description: str = Field(description="Short description of the item used to answer the question")

class RAGGenerationResponse(BaseModel):
    answer: str = Field(description="Answer to the question.")
    references: list[RAGUsedContext] = Field(description="List of items to answer the question.")

In [10]:
def get_embedding(text, model='text-embedding-3-small'):

    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding

def retrieve_data(query, qdrant_client, k=5):
    query_embedding = get_embedding(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []
    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }

def process_context(context):
    
    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context

def build_prompt(preprocessed_context, question):

    prompt = f"""
    You are a shopping assistant that can answer questions about the products in stock.

    You will be given a question and a list of context.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use word context and refer to it as the available products.
    - Do not use markdown formatting.

    Context:
    {preprocessed_context}

    Question:
    {question}
    """

    return prompt

def generate_answer(prompt):

    response, raw_response = client.create_with_completion(
    model="gpt-5.4-mini",
    messages=[
        {"role": "system", "content": prompt}
    ],
    reasoning={"effort": "none"},
    response_model=RAGGenerationResponse
)

    return response

def RAG_pipeline(question, top_k=5):

    qdrant_client = QdrantClient(url="http://localhost:6333")

    retrieved_context = retrieve_data(question, qdrant_client, top_k)
    preprocess_context = process_context(retrieved_context)
    prompt = build_prompt(preprocess_context, question)
    answer = generate_answer(prompt)

    final_result = {
        "data_object": answer,
        "answer": answer.answer,
        "references": answer.references,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"],
        "similarity_scores": retrieved_context["similarity_scores"],
    }

    return final_result

In [11]:
output = RAG_pipeline("Can I buy a fiction book?")

In [12]:
output

{'data_object': RAGGenerationResponse(answer='Yes — the available products include several fiction books, such as The Man His Means & His Methods, Devil of Dublin, Shorn, Monsters of Midlife, and Faultless Love.', references=[RAGUsedContext(id='B0BLFT3P9C', description='Fiction novel The Man His Means & His Methods'), RAGUsedContext(id='B0BFW7MV1C', description='Dark Irish mafia romance Devil of Dublin'), RAGUsedContext(id='B09XGZV7YT', description='Fantasy novel Shorn'), RAGUsedContext(id='B0B6LSBPR1', description="Paranormal women's fiction trilogy Monsters of Midlife"), RAGUsedContext(id='1737747227', description='Small town second chance romance Faultless Love')]),
 'answer': 'Yes — the available products include several fiction books, such as The Man His Means & His Methods, Devil of Dublin, Shorn, Monsters of Midlife, and Faultless Love.',
 'references': [RAGUsedContext(id='B0BLFT3P9C', description='Fiction novel The Man His Means & His Methods'),
  RAGUsedContext(id='B0BFW7MV1C'